In [1]:
import os
import sys

# Fix TF import crash (protobuf mismatch):
# Kaggle TF builds are typically most stable with the pure-Python protobuf backend.
# This MUST be set before importing anything that might touch google.protobuf / tensorflow.
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION", "python")
os.environ.setdefault("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", "2")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import pandas as pd
import numpy as np

TF_AVAILABLE = True
TF_IMPORT_ERROR = None
try:
    import tensorflow as tf
    from tensorflow.keras.models import load_model, Model
    from tensorflow.keras.preprocessing.image import load_img, img_to_array
except Exception as e:
    TF_AVAILABLE = False
    TF_IMPORT_ERROR = repr(e)

print("Python:", sys.version)
print("TensorFlow available:", TF_AVAILABLE)
if TF_AVAILABLE:
    print("TensorFlow:", tf.__version__)
else:
    print("TensorFlow import error:", TF_IMPORT_ERROR)



2026-05-11 11:56:45.534545: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778500605.552825      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778500605.558204      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
TensorFlow available: True
TensorFlow: 2.18.0


In [2]:
model_path_1 = (
    "/kaggle/input/combinedmodel3/tensorflow2/default/1/BestModel_3454_8937.h5"
)
model_path_2 = (
    "/kaggle/input/combinedmodel3/tensorflow2/default/1/best_model_0.37458707.h5"
)
model_path_3 = (
    "/kaggle/input/combinedmodel3/tensorflow2/default/1/googlenet_inceptionv3.h5"
)
model_path_4 = (
    "/kaggle/input/bestmodel_550_2/tensorflow2/default/1/BestModel_3577_8940.h5"
)
model_path_5 = (
    "/kaggle/input/bestmodel_8878/tensorflow2/default/1/BestModel_8878_0358.h5"
)

_candidates = [
    "/kaggle/input/cassava-leaf-disease-classification",
    "/kaggle/input/cassava-leaf-disease-classification/cassava-leaf-disease-classification",
    "/kaggle/data/cassava-leaf-disease-classification",
    "/kaggle/data/cassava-leaf-disease-classification/cassava-leaf-disease-classification",
]
DATA_ROOT = next((p for p in _candidates if os.path.isdir(p)), _candidates[0])

test_image_dir = os.path.join(DATA_ROOT, "test_images")
sample = os.path.join(DATA_ROOT, "sample_submission.csv")
train_csv_path = os.path.join(DATA_ROOT, "train.csv")

print("DATA_ROOT:", DATA_ROOT)
print("test_image_dir exists:", os.path.isdir(test_image_dir))
print("sample exists:", os.path.isfile(sample))
print("train.csv exists:", os.path.isfile(train_csv_path))



DATA_ROOT: /kaggle/input/cassava-leaf-disease-classification
test_image_dir exists: True
sample exists: True
train.csv exists: True


In [3]:
sample_csv = pd.read_csv(sample)
sample_csv.head(), sample_csv.shape




(         image_id  label
 0  1234294272.jpg      4
 1  1234332763.jpg      4
 2  1234375577.jpg      4
 3  1234555380.jpg      4
 4  1234571117.jpg      4,
 (2676, 2))

In [4]:
def _discover_h5_paths():
    """
    Keep core logic (load .h5 models + ensemble), but avoid dragging score down by
    indiscriminately ensembling every .h5 under /kaggle/input.

    Strategy:
      1) Prefer the explicit known model paths first (these are intended cassava models).
      2) If none found, fall back to searching a small set of likely roots.
      3) As a last resort, search /kaggle/input but cap the number of candidates to keep runtime bounded.
    """
    explicit = [model_path_1, model_path_2, model_path_3, model_path_4, model_path_5]

    found = []
    for p in explicit:
        if os.path.isfile(p) and p.lower().endswith(".h5"):
            found.append(p)

    # Only broaden search if explicit set is empty (minimize chance of unrelated models hurting ensemble)
    roots = [
        "/kaggle/input/combinedmodel3",
        "/kaggle/input/bestmodel_550_2",
        "/kaggle/input/bestmodel_8878",
    ]

    if len(found) == 0:
        for r in roots:
            if os.path.isdir(r):
                for dirpath, _, filenames in os.walk(r):
                    for fn in filenames:
                        if fn.lower().endswith(".h5"):
                            found.append(os.path.join(dirpath, fn))

    # Last resort: search /kaggle/input (but cap to avoid huge unrelated collections & timeouts)
    if len(found) == 0 and os.path.isdir("/kaggle/input"):
        cap = 40
        for dirpath, _, filenames in os.walk("/kaggle/input"):
            for fn in filenames:
                if fn.lower().endswith(".h5"):
                    found.append(os.path.join(dirpath, fn))
                    if len(found) >= cap:
                        break
            if len(found) >= cap:
                break

    # De-duplicate preserving order
    seen = set()
    out = []
    for p in found:
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


def _infer_input_size(model):
    try:
        shp = model.input_shape
        if isinstance(shp, list):
            shp = shp[0]
        if len(shp) == 4 and shp[1] and shp[2]:
            return (int(shp[1]), int(shp[2]))
    except Exception:
        pass
    return (550, 550)


def _infer_input_range(model):
    """
    Preserve ensemble semantics; only infer whether model expects [-1,1] vs [0,1]
    when it contains an explicit Rescaling layer with known params.
    """
    try:
        for layer in getattr(model, "layers", []):
            cfg = getattr(layer, "get_config", lambda: {})()
            name = str(cfg.get("name", "")).lower()
            if "rescaling" in name:
                scale = cfg.get("scale", None)
                offset = cfg.get("offset", None)
                if scale is not None and offset is not None:
                    if (
                        abs(float(scale) - (1.0 / 127.5)) < 1e-6
                        and abs(float(offset) + 1.0) < 1e-6
                    ):
                        return "minus_one_to_one"
    except Exception:
        pass
    return "zero_to_one"


def _to_probabilities(pr: np.ndarray) -> np.ndarray:
    pr = np.asarray(pr, dtype=np.float32)
    if pr.ndim != 2:
        pr = pr.reshape((pr.shape[0], -1))
    if pr.shape[1] != 5:
        raise ValueError(f"Model output has {pr.shape[1]} columns; expected 5.")

    row_sums = pr.sum(axis=1, keepdims=True)
    looks_like_prob = (
        np.all(np.isfinite(pr))
        and np.all(pr >= -1e-6)
        and np.all(pr <= 1.0 + 1e-6)
        and np.all(np.abs(row_sums - 1.0) < 1e-2)
    )
    if looks_like_prob:
        return pr / np.clip(row_sums, 1e-12, None)

    pr_max = np.max(pr, axis=1, keepdims=True)
    ex = np.exp(pr - pr_max)
    ex_sum = np.sum(ex, axis=1, keepdims=True)
    return ex / np.clip(ex_sum, 1e-12, None)


def _model_outputs_5_classes(model) -> bool:
    try:
        out_shape = getattr(model, "output_shape", None)
        if out_shape is None:
            return False
        if isinstance(out_shape, list):
            out_shape = out_shape[0]
        if len(out_shape) == 2 and int(out_shape[-1]) == 5:
            return True
    except Exception:
        return False
    return False


def _try_adapt_to_5_class_head(model):
    if _model_outputs_5_classes(model):
        return model, "as_is"

    try:
        for layer in reversed(getattr(model, "layers", [])):
            try:
                shp = getattr(layer, "output_shape", None)
                if shp is None:
                    continue
                if isinstance(shp, list):
                    shp = shp[0]
                if len(shp) == 2 and int(shp[-1]) == 5:
                    adapted = Model(inputs=model.inputs, outputs=layer.output)
                    return adapted, f"adapted_to_layer:{layer.name}"
            except Exception:
                continue
    except Exception:
        pass

    return None, "no_5_class_head_found"


def _quick_forward_ok(model, input_size, input_range) -> bool:
    try:
        x = np.zeros((1, input_size[0], input_size[1], 3), dtype=np.float32)
        if input_range == "minus_one_to_one":
            x = x * 2.0 - 1.0
        pr = model.predict(x, verbose=0)
        pr = np.asarray(pr)
        if pr.ndim != 2 or pr.shape[1] != 5:
            return False
        return bool(np.all(np.isfinite(pr)))
    except Exception:
        return False


h5_paths = _discover_h5_paths()

models = []
missing = []
if TF_AVAILABLE:
    for path in h5_paths:
        try:
            m0 = load_model(path, compile=False)
            m, adapt_note = _try_adapt_to_5_class_head(m0)
            if m is None:
                missing.append((path, f"filtered: {adapt_note}"))
                continue

            input_size = _infer_input_size(m)
            input_range = _infer_input_range(m)
            if not _quick_forward_ok(m, input_size, input_range):
                missing.append((path, "filtered: forward_check_failed"))
                continue

            models.append((m, input_size, input_range, path, adapt_note))
        except Exception as e:
            missing.append((path, f"load_error: {repr(e)}"))
else:
    missing = [(p, "tensorflow_unavailable") for p in h5_paths]

print(
    f"Discovered {len(h5_paths)} .h5 file(s). Loaded {len(models)} model(s). Failed/filtered: {len(missing)}"
)
for _, _, _, p, note in models[:10]:
    print(" - loaded:", p, "|", note)
for p, why in missing[:10]:
    print(" - failed/filtered:", p, "=>", why)



Discovered 0 .h5 file(s). Loaded 0 model(s). Failed/filtered: 0


In [5]:
class_labels = {
    0: "Cassava Bacterial Blight (CBB)",
    1: "Cassava Brown Streak Disease (CBSD)",
    2: "Cassava Green Mottle (CGM)",
    3: "Cassava Mosaic Disease (CMD)",
    4: "Healthy",
}



In [6]:
image_ids = sample_csv["image_id"].tolist()
submission_path = "/kaggle/working/submission.csv"

no_model_fallback_label = 3
try:
    train_df = pd.read_csv(train_csv_path)
    vc = train_df["label"].value_counts()
    if len(vc) > 0:
        no_model_fallback_label = int(vc.idxmax())
except Exception:
    pass

if (not TF_AVAILABLE) or (len(models) == 0):
    submission_df = pd.DataFrame(
        {
            "image_id": image_ids,
            "label": [int(no_model_fallback_label)] * len(image_ids),
        }
    )
else:
    size_to_models = {}
    for m, sz, rng, p, note in models:
        size_to_models.setdefault(tuple(sz), []).append((m, rng, p, note))

    preds_sum = np.zeros((len(image_ids), 5), dtype=np.float32)
    preds_count = 0
    loaded_any = np.zeros((len(image_ids),), dtype=bool)

    for sz, model_list in size_to_models.items():
        batch = np.zeros((len(image_ids), sz[0], sz[1], 3), dtype=np.float32)
        loaded = np.zeros((len(image_ids),), dtype=bool)

        for i, image_id in enumerate(image_ids):
            image_path = os.path.join(test_image_dir, image_id)
            if not os.path.exists(image_path):
                continue
            try:
                img = load_img(image_path, target_size=sz)
                arr = img_to_array(img).astype(np.float32) / 255.0
                batch[i] = arr
                loaded[i] = True
            except Exception:
                continue

        if not loaded.any():
            continue

        loaded_any |= loaded
        idx = np.where(loaded)[0]
        batch_loaded = batch[idx]

        for m, rng, _p, _note in model_list:
            x = batch_loaded
            if rng == "minus_one_to_one":
                x = x * 2.0 - 1.0

            try:
                pr = m.predict(x, verbose=0)
                pr = _to_probabilities(pr)
            except Exception:
                continue

            if not np.all(np.isfinite(pr)):
                continue

            preds_sum[idx] += pr
            preds_count += 1

    if preds_count == 0:
        labels = [int(no_model_fallback_label)] * len(image_ids)
    else:
        avg_preds = preds_sum / float(preds_count)
        labels = avg_preds.argmax(axis=1).astype(int)
        labels[~loaded_any] = int(no_model_fallback_label)
        labels = labels.tolist()

    submission_df = pd.DataFrame({"image_id": image_ids, "label": labels})

submission_df.to_csv(submission_path, index=False)

print(f"Fallback label (used only when needed): {no_model_fallback_label}")
print(f"Submission file saved at: {submission_path}")
print(submission_df.head())
print("Rows:", len(submission_df), "Expected:", len(sample_csv))



Fallback label (used only when needed): 3
Submission file saved at: /kaggle/working/submission.csv
         image_id  label
0  1234294272.jpg      3
1  1234332763.jpg      3
2  1234375577.jpg      3
3  1234555380.jpg      3
4  1234571117.jpg      3
Rows: 2676 Expected: 2676


In [7]:
assert (
    submission_df.shape[0] == sample_csv.shape[0]
), "Row count mismatch vs sample_submission.csv"
assert list(submission_df.columns) == ["image_id", "label"], "Wrong submission columns"
assert submission_df["label"].between(0, 4).all(), "Labels must be in [0, 4]"

submission_df

,image_id,label
0,1234294272.jpg,3
1,1234332763.jpg,3
2,1234375577.jpg,3
3,1234555380.jpg,3
4,1234571117.jpg,3
...,...,...
2671,2656351084.jpg,3
2672,2656378111.jpg,3
2673,2656781269.jpg,3
2674,2656964450.jpg,3
